# Phase 1b: How many VMs can we afford?

A bigger sample makes the results more convincing, but every extra VM makes training, tuning (Phase 4) and the five-seed stability test (Phase 6) slower. Instead of guessing, this notebook **measures** how long each model takes on this exact Colab machine and projects the time for the whole project at 50, 75 and 100 VMs.

How it works:
1. Build features for the first 10 and then the first 20 main-group VMs (a small preview of what Phase 2 will do).
2. Train each model once with default settings at both sizes and time it. For the LSTM we time single epochs.
3. From the 10 → 20 VM jump, estimate how training time grows with data size, then project to 50/75/100 VMs.
4. Add up what every later phase will need (tuning grids, energy run, 5 seeds) into one estimate per sample size.

**Nothing here is a result for the paper.** It's only a planning tool, and the models trained here are thrown away.

**Before running:** check the runtime is CPU (`Runtime → Change runtime type → CPU`). We use CPU for every model so that the time and energy comparisons are fair. It takes about 10–15 minutes.

## 0. Setup

In [ ]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

In [ ]:
import gc, os, pickle, platform
import numpy as np
import pandas as pd
import model_definitions as M
importlib.reload(M)

log = U.get_logger('phase01b_feasibility')

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    ram_gb = float('nan')
log.info(f'CPU cores visible: {os.cpu_count()} | RAM: {ram_gb:.1f} GB | processor: {platform.processor() or platform.machine()}')

## 1. A small preview of the data

We take the main group's first 20 VMs (by `pool_rank`) from Phase 1's output and build features the same way Phase 2 will: lags, rolling window, hour/minute and the next-step target. Each VM is then split 80/20 by time.

In [ ]:
clean = pd.read_parquet(C.DATA_PROCESSED / 'cleaned_vms.parquet')
main = clean[clean['pool'] == 'main']

sizes = [10, 20]
data = {}
for n in sizes:
    feat = U.build_features(main[main['pool_rank'] < n])
    tr, te = U.chrono_split(feat)
    data[n] = (feat, tr, te)
    log.info(f'{n} VMs -> {len(tr):,} training rows, {len(te):,} test rows')

rows_per_vm = len(data[20][1]) / 20
log.info(f'Average training rows per VM: {rows_per_vm:,.0f}')

## 2. Timing each model

Each model is trained once at each size, using all CPU cores. We also check how much space each trained model takes when saved, because a full-depth Random Forest can be surprisingly large on disk.

In [ ]:
timings = []
for n in sizes:
    feat, tr, te = data[n]
    X, y = tr[U.FEATURE_COLUMNS], tr[C.TARGET]
    for name in M.TABULAR_MODELS:
        model = M.make_model(name)
        with U.Timer() as t:
            model.fit(X, y)
        with U.Timer() as tp:
            pred = model.predict(te[U.FEATURE_COLUMNS])
        size_mb = len(pickle.dumps(model)) / 1e6
        r2 = U.regression_metrics(te[C.TARGET], pred)['R2']
        timings.append({'model': name, 'vms': n, 'fit_sec': t.seconds, 'predict_sec': tp.seconds,
                        'model_mb': size_mb, 'r2_preview': r2})
        log.info(f'{n:>3} VMs | {name:<17} fit {t.seconds:7.2f}s | predict {tp.seconds:5.2f}s | '
                 f'{size_mb:8.1f} MB | R2 {r2:.3f}')
        del model; gc.collect()

    # LSTM (provisional settings): time 2 epochs, use the second one (the first includes warm-up)
    X_seq, y_seq, idx = M.make_sequences(feat, M.LSTM_DEFAULTS['seq_len'])
    train_mask = ~U.test_mask(feat)[idx]          # windows whose target row is in the training period
    _, hist = M.train_lstm(X_seq[train_mask], y_seq[train_mask], max_epochs=2, log_fn=log.info)
    timings.append({'model': 'LSTM', 'vms': n, 'fit_sec': hist[-1]['seconds'], 'predict_sec': np.nan,
                    'model_mb': np.nan, 'r2_preview': np.nan})
    log.info(f'{n:>3} VMs | LSTM (1 epoch)     {hist[-1]["seconds"]:7.2f}s')

timings = pd.DataFrame(timings)
timings

## 3. Projecting to 50, 75 and 100 VMs

Training time doesn't always grow in a straight line with the amount of data. Tree models, for example, have to sort values, which costs a little more than linear. So we look at how much slower each model got when the data doubled (10 → 20 VMs) and assume the same growth rate holds further up:

`time(n VMs) ≈ time(20 VMs) × (n / 20) ^ b`, where `b` is measured from the doubling. It's clipped to between 0.8 and 1.5 so that timing noise on very fast models (Linear Regression finishes in well under a second) can't produce a silly projection.

Then we add up what the later phases actually do with each model:

| Phase | Work per model |
|---|---|
| 3: training | 1 fit |
| 4: tuning | grid size × 3 folds (TimeSeriesSplit trains on ¼, ½ and ¾ of the data, so ≈ 1.5 full fits per grid point) + 1 refit, multiplied by a cost factor because some grid points use more trees than the defaults |
| 5: energy | 1 fit (retrained with CodeCarbon running) |
| 6: stability | 5 fits (one per seed) |

For the LSTM we assume up to 20 epochs per training run and a small search of 6 settings in Phase 4 (to be confirmed).

In [ ]:
from sklearn.model_selection import ParameterGrid
grid_points = {name: len(ParameterGrid(grid)) for name, grid in M.PARAM_GRIDS.items()}
# Rough cost of an average grid point relative to the default model
grid_cost = {'Decision Tree': 1.0, 'Random Forest': 1.2, 'XGBoost': 2.0, 'LightGBM': 2.0}
LSTM_EPOCHS, LSTM_TUNING_RUNS = M.LSTM_DEFAULTS['max_epochs'], len(M.LSTM_TUNING_GRID)

def fits_needed(name):
    tuning = 0 if name == 'Linear Regression' else (
        LSTM_TUNING_RUNS if name == 'LSTM' else grid_points[name] * 1.5 * grid_cost[name] + 1)
    per_run = LSTM_EPOCHS if name == 'LSTM' else 1
    return {'p3': per_run, 'p4': tuning * per_run, 'p5': per_run, 'p6': 5 * per_run}

projection = []
for name, g in timings.groupby('model', sort=False):
    t10, t20 = g.set_index('vms')['fit_sec'].loc[[10, 20]]
    b = float(np.clip(np.log(max(t20, 1e-3) / max(t10, 1e-3)) / np.log(2), 0.8, 1.5))
    for n in [50, 75, 100]:
        t_fit = t20 * (n / 20) ** b
        f = fits_needed(name)
        projection.append({'model': name, 'vms': n, 'growth_b': b, 'one_fit_min': t_fit / 60,
                           **{f'{k}_min': v * t_fit / 60 for k, v in f.items()}})
projection = pd.DataFrame(projection)

summary = (projection.groupby('vms')[['p3_min', 'p4_min', 'p5_min', 'p6_min']].sum())
summary['total_hours'] = summary.sum(axis=1) / 60
summary = summary.rename(columns={'p3_min': 'Phase 3 (min)', 'p4_min': 'Phase 4 tuning (min)',
                                  'p5_min': 'Phase 5 energy (min)', 'p6_min': 'Phase 6 seeds (min)',
                                  'total_hours': 'Total (hours)'})
summary.index.name = 'VMs in main group'

U.log_df(log, projection.round(2), 'PER-MODEL PROJECTION')
U.log_df(log, summary.round(1), 'PROJECTED COMPUTE TIME PER SAMPLE SIZE')
summary.round(1)

In [ ]:
# Which model dominates the cost at 100 VMs?
projection[projection.vms == 100].set_index('model')[['one_fit_min', 'p4_min', 'p6_min']].round(1)

In [ ]:
rf = timings[(timings.model == 'Random Forest') & (timings.vms == 20)]['model_mb'].iloc[0]
log.info(f'Saved Random Forest size at 20 VMs: {rf:.0f} MB -> roughly {rf * 5:.0f} MB at 100 VMs')
log.info('FEASIBILITY CHECK COMPLETE')

## What to do with these numbers

Send the **"Projected compute time"** table (or just a screenshot of it) back to the team or to Claude. As a rough guide:

- Free Colab stops a session after about 12 hours at most, and earlier if the tab sits idle. **No single notebook should need more than about 3–4 hours.** Phase 4 (tuning) is usually the longest, and it saves its progress after each model so a disconnect doesn't lose everything.
- If 100 VMs keeps every phase comfortably under that, use 100 + 100.
- If not, use 75 or 50, or keep 100 VMs and make the Random Forest grid smaller (it's usually the model that blows up the time).

The sample size is then set in `scripts/config.py` (`N_TRAIN_VMS`, `N_GEN_VMS`) and used by every phase from 2 onwards.